# Experiment: Long Sequence Spatial Annotation

Create desk-plane geometry annotations for the long random-trajectory smell sessions. The default batch is the six-session **2026-08-03** recording batch; the five annotated 2026-07-30 sessions remain selectable for inspection.

This notebook does not reuse geometry from preliminary analyses. Each saved `spatial_annotation.json` is derived from clicks made here.

## Annotation contract

- Use the synchronized `pcnose_flag == 2` interval as the default usable range; shorten it only for a concrete data-quality reason.
- Choose one sharp, unobstructed raw frame. The calibrated desk view uses the desk-tag pose from that same CSV row, so moving the camera during a session is supported.
- Click the paper corners in this order: **top-left, top-right, bottom-right, bottom-left**.
- For every odor strip, click its four **outer** corners in cyclic order. For an X layout, annotate both complete strips independently, including the portion hidden at the crossing; do not add special intersection clicks.
- Once both X strips are complete, their desk-plane intersection is shown in yellow as QC. `Overlap top` optionally records which physical strip is above the other; leave it unknown if the video does not establish this.
- `Undo last click` removes exactly the most recent point. `Reset current` clears only the selected paper/source feature.
- `Hide annotation overlay` hides all paper/source points, outlines, labels, overlap shading, and raw-frame reprojections. Image clicks are disabled while the overlay is hidden; press `Show annotation overlay` to resume editing.
- Saving is final for that session and overwrites only that session's JSON. Selecting the August batch therefore leaves all July annotations untouched.

The saved overlap metadata states that future area-class evaluation should mask the ambiguous crossing. The source polygons themselves remain complete and can still define distance-to-source targets.

In [ ]:
from pathlib import Path
import sys

import ipywidgets as widgets
from IPython.display import display

candidate_roots = [Path.cwd(), *Path.cwd().parents]
repo_root = next(
    (path for path in candidate_roots if (path / "analysis" / "spatial_sequence").is_dir()),
    None,
)
if repo_root is None:
    raise RuntimeError("Run this notebook from inside the realsense-apriltag repository")
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

get_ipython().run_line_magic("matplotlib", "widget")

from analysis.spatial_sequence.annotation import SpatialAnnotationTool, annotation_status_table
from analysis.spatial_sequence.core import discover_august_sessions, discover_sessions

batch_sessions = {
    "2026-08-03": discover_august_sessions(),
    "2026-07-30": discover_sessions(),
}
batch_picker = widgets.Dropdown(
    options=[
        ("2026-08-03 — six new sessions", "2026-08-03"),
        ("2026-07-30 — five existing sessions", "2026-07-30"),
    ],
    value="2026-08-03",
    description="Batch",
    layout=widgets.Layout(width="560px"),
)
display(batch_picker)

## Select and inspect a batch

Choose the batch above, then run the next cell. If you change the dropdown later, rerun this cell and the annotation-tool cell below.

In [ ]:
selected_batch = str(batch_picker.value)
sessions = batch_sessions[selected_batch]
print(f"Selected {selected_batch}: {len(sessions)} sessions")
display(annotation_status_table(sessions))

## Interactive annotation

For each session:

1. Confirm or adjust the contiguous usable row range.
2. Move `Reference row` until both paper and strip boundaries are as clear as possible in the raw and calibrated views.
3. Select `paper` and click its four corners in the required order.
4. Select each source and click its four outer corners cyclically.
5. Inspect the outlines in **both** the calibrated and raw-frame views. Use `Hide annotation overlay` to compare against the unobstructed frame, then show it again before editing.
6. For a crossing layout, inspect the yellow overlap preview and optionally choose its top strip.
7. Save the JSON and continue to the next session.

The points are expressed in the persistent desk reference frame, not in a fixed camera frame.

In [ ]:
if "tool" in globals():
    tool.close()

tool = SpatialAnnotationTool(sessions)
display(tool.widget)

## Batch status

Rerun this cell after saving. It reports completion for the currently selected batch without changing any annotation.

In [ ]:
final_status = annotation_status_table(sessions)
display(final_status)

valid_count = int(final_status["annotation"].eq("valid").sum())
if valid_count == len(sessions):
    print(f"Complete: all {len(sessions)} {selected_batch} annotations are valid.")
else:
    print(
        f"Incomplete: {valid_count}/{len(sessions)} {selected_batch} annotations are valid; "
        "continue with the sessions marked missing or invalid."
    )

## Notes

- The displayed desk view is a calibrated **single frame**, not a temporal median.
- The right panel is the matching raw frame with the current desk-plane annotations reprojected for QC.
- The overlay toggle affects both panels and only changes display state; it never deletes annotation points.
- A source-hash mismatch is reported when an annotation is later loaded, but does not silently replace geometry.
- Close the widget when finished with `tool.close()` if the notebook kernel will remain running.